# DPO: concise professional email rewriting

This notebook is a **thin demo**: every step calls into the tested `geap_tuning`
package. It mirrors [`examples/run_preference_email.py`](../examples/run_preference_email.py).

Where our other DPO demo tunes support-reply *warmth*, this teaches **concise +
professional** email rewriting and records a measured **before → after** lift. A
blind A/B autorater (a prompt **distinct** from the generator) picks the more
professional, concise version; both preference completions carry the *same
facts*, so the judge grades style, not content. An objective compression ratio
backs up the win-rate.

> **Requires live GCP and incurs tuning cost** (one preference-tuning job). Have
> a real `.env` and `gcloud auth` in place.

In [ ]:
from geap_tuning.config import genai_client, load_config

BASE_MODEL = "gemini-2.5-flash"
JUDGE_MODEL = "gemini-2.5-flash"
cfg = load_config()
client = genai_client(cfg)
cfg

## 1. Build the preference dataset and stage it to GCS

Each record is a `(draft, preferred, dispreferred)` triple — hand-authored so
the preferred rewrite is professional **and** materially shorter than the
dispreferred one (a concision invariant the unit tests enforce).

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.preference.email import (
    EMAIL_DRAFTS,
    SYSTEM_INSTRUCTION,
    build_preference_dataset,
    build_preference_records,
    split_dataset,
)

paths = build_preference_dataset("../datasets/preference_concise_email")
train_uri = upload_file(paths["train"], f"{cfg.bucket}/preference_concise_email/train.jsonl")
val_uri = upload_file(paths["val"], f"{cfg.bucket}/preference_concise_email/val.jsonl")

_, _, test_triples = split_dataset(EMAIL_DRAFTS)
test_records = build_preference_records(test_triples)
print(f"{len(EMAIL_DRAFTS)} triples, {len(test_records)} held out")

## 2. The blind A/B judge

The judge prompt is **different** from the generator's `SYSTEM_INSTRUCTION` and
reads only the first letter of the verdict. Candidate A is always the model under
test; B is the dispreferred reference.

In [ ]:
from geap_tuning.inference import generate

_JUDGE_PROMPT = (
    "You are judging two versions of the same work email. Pick the version that "
    "is more professional and concise (clear, brief, free of filler and hedging) "
    "while keeping the same information. Answer with only the single letter 'A' "
    "or 'B'.\n\n"
    "Original draft: {user}\n\nEmail A: {a}\n\nEmail B: {b}\n\nBetter email:"
)


def judge_fn(draft: str, cand_a: str, cand_b: str) -> str:
    verdict = generate(client, JUDGE_MODEL, _JUDGE_PROMPT.format(user=draft, a=cand_a, b=cand_b))
    return verdict[:1].upper()

## 3. Score the untuned base (the "before")

`run_email_eval` generates one rewrite per draft, then reports the autorater
`win_rate` (vs. the dispreferred reference) alongside `mean_compression` (rewrite
/ draft word ratio; < 1 means shorter).

In [ ]:
from geap_tuning.preference.email_eval import run_email_eval

base = run_email_eval(
    test_records,
    generate_fn=lambda draft: generate(
        client, BASE_MODEL, draft, system_instruction=SYSTEM_INSTRUCTION
    ),
    judge_fn=judge_fn,
)
print(
    f"BASE win_rate={base['win_rate']:.3f} "
    f"mean_compression={base['mean_compression']:.2f} (n={base['n']})"
)

## 4. Launch the preference-tuning job and wait

In [ ]:
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_endpoint, wait_for_tuning_job
from geap_tuning.preference.tune import launch_preference_job

DISPLAY_NAME = "geap-dpo-concise-email"

job = find_tuning_job_by_display_name(client, DISPLAY_NAME)
if job is None:
    job = launch_preference_job(
        client,
        train_uri=train_uri,
        val_uri=val_uri,
        display_name=DISPLAY_NAME,
        base_model=BASE_MODEL,
        labels=cfg.labels,
    )
job = wait_for_tuning_job(client, job.name)
endpoint = tuned_endpoint(job)
endpoint

## 5. Score the tuned endpoint (the "after") and report the lift

In [ ]:
tuned = run_email_eval(
    test_records,
    generate_fn=lambda draft: generate(
        client, endpoint, draft, system_instruction=SYSTEM_INSTRUCTION
    ),
    judge_fn=judge_fn,
)
print(
    f"TUNED win_rate={tuned['win_rate']:.3f} "
    f"mean_compression={tuned['mean_compression']:.2f} (n={tuned['n']})"
)
print(
    f"LIFT win_rate {base['win_rate']:.3f}->{tuned['win_rate']:.3f}; "
    f"compression {base['mean_compression']:.2f}->{tuned['mean_compression']:.2f}"
)